# Matching and Matched Inference: Singapore FY2023
Implements proposal Section 4(2)–(4): matching, sign test, permutation test, bootstrap CI and matched quantile gaps.

Load the cleaned data

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TAB_DIR = ROOT / "outputs" / "tables"
FIG_DIR = ROOT / "outputs" / "figures"
TAB_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = ROOT / "data" / "processed" / "orbis_singapore_fy2023_clean.parquet"
df = pd.read_parquet(DATA_FILE)

print("File:", DATA_FILE.name)
print("Shape:", df.shape)
print(df.columns.tolist())

File: orbis_singapore_fy2023_clean.parquet
Shape: (48995, 32)
['Company name Latin alphabet', 'Inactive', 'Quoted', 'branch', 'OwnData', 'Woco', 'country_code', 'nace4', 'consolidation_code', 'bvd_id', 'incorporation_date', 'assets_th_usd', 'Total assets th USD 2022', 'revenue_th_usd', 'Operating revenue (Turnover) th USD 2022', 'employees', 'Number of employees 2022', 'ish_bvd_id', 'guo_bvd_id', 'ISH - Name', 'GUO - Name', 'SH - BvD Independence Indicator', 'listing_status', 'status', 'Closing date Last avail. yr', 'closing_date_2023', 'Closing date 2022', '_source_file', 'listed', 'assets_usd', 'log_assets', 'nace2']


Build the matching variables

In [2]:
# Listing status as text, from the cleaning script's 1/0 flag
df["listing_status"] = df["listed"].map({1: "Listed", 0: "Unlisted"})

# Incorporation year: the cleaned file lost year-only dates (e.g. "2006"),
# so take the year from the original text in the merged file, matched by firm ID
MERGED_FILE = ROOT / "data" / "interim" / "orbis_singapore_fy2023_merged.parquet"
raw = pd.read_parquet(MERGED_FILE, columns=["BvD ID number", "Date of incorporation"])
raw_year = pd.to_numeric(
    raw["Date of incorporation"].astype("string").str.extract(r"(\d{4})", expand=False),
    errors="coerce")
year_by_id = pd.Series(raw_year.to_numpy(), index=raw["BvD ID number"].astype("string"))
print("Duplicate IDs in merged file:", year_by_id.index.duplicated().sum())
year_by_id = year_by_id[~year_by_id.index.duplicated()]
df["incorporation_year"] = df["bvd_id"].astype("string").map(year_by_id)

# Actual age in FY2023
df["firm_age"] = 2023 - df["incorporation_year"]
df.loc[df["firm_age"] < 0, "firm_age"] = np.nan
df["age_band"] = pd.cut(df["firm_age"], bins=[-1, 4, 9, 19, float("inf")],
                        labels=["0–4", "5–9", "10–19", "20+"])

# Account type: C (consolidated) vs U (unconsolidated)
df["consolidated"] = df["consolidation_code"].astype("string").str.strip().str.upper().str[0]
df["consolidated"] = df["consolidated"].where(df["consolidated"].isin(["C", "U"]))

# Analysis sample: positive assets only (log_assets = ln(assets_usd), checked = 0 difference)
eda = df[df["listing_status"].notna() & (df["assets_usd"] > 0)].copy()

print("\nFirms by status:\n", eda["listing_status"].value_counts())
print("\nMissing matching variables:")
print(eda.groupby("listing_status")[["nace2", "firm_age", "age_band", "consolidated", "log_assets"]]
         .apply(lambda g: g.isna().sum()))

Duplicate IDs in merged file: 0

Firms by status:
 listing_status
Unlisted    47949
Listed        496
Name: count, dtype: int64

Missing matching variables:
                nace2  firm_age  age_band  consolidated  log_assets
listing_status                                                     
Listed              0         0         0             0           0
Unlisted            0         0         0             0           0


Exclude subsidiaries of listed firms

Private firms whose immediate shareholder (ISH) or global ultimate owner (GUO) is a sampled listed firm are excluded, as specified in the proposal. Ownership links at intermediate levels are not captured, which is a limitation.

In [3]:
# Exclude private subsidiaries of sampled listed firms (proposal Section 3)
listed_ids = set(eda.loc[eda["listing_status"] == "Listed", "bvd_id"].astype(str))
owner_is_listed = (eda["ish_bvd_id"].astype("string").isin(listed_ids).fillna(False)
                   | eda["guo_bvd_id"].astype("string").isin(listed_ids).fillna(False))
is_sub = (eda["listing_status"] == "Unlisted") & owner_is_listed

owners = pd.concat([eda.loc[is_sub, "ish_bvd_id"], eda.loc[is_sub, "guo_bvd_id"]]).dropna().astype(str)
print("Private firms whose ISH or GUO is a sampled listed firm:", int(is_sub.sum()))
print("Distinct listed parents involved:", owners[owners.isin(listed_ids)].nunique())

eda = eda[~is_sub].copy()
print("\nFirms by status after exclusion:\n", eda["listing_status"].value_counts())

Private firms whose ISH or GUO is a sampled listed firm: 1997
Distinct listed parents involved: 365

Firms by status after exclusion:
 listing_status
Unlisted    45952
Listed        496
Name: count, dtype: int64


## Matching
For each listed firm, up to three private firms are selected without replacement from the same NACE 2-digit industry, age band and account type (C/U), choosing those closest in actual firm age. Ties are broken randomly (seed 607). Country and financial year are constant (Singapore, FY2023).

In [4]:
KEYS = ["nace2", "age_band", "consolidated"]
ID = "bvd_id"
MAX_CONTROLS = 3

def run_matching(data, max_age_gap=None, seed=607):
    """Round-robin nearest-age matching, up to MAX_CONTROLS, without replacement.
    max_age_gap: if set, controls must be within this many years of the listed firm."""
    rng = np.random.default_rng(seed)
    base = data.dropna(subset=["log_assets", "firm_age"] + KEYS)
    L_all = base[base["listing_status"] == "Listed"]
    pool = base[base["listing_status"] == "Unlisted"]
    pool_groups = {k: g for k, g in pool.groupby(KEYS, observed=True)}
    sets, unmatched, n_sets = [], [], 0

    for key, L in L_all.groupby(KEYS, observed=True):
        L = L.sample(frac=1, random_state=int(rng.integers(1_000_000)))
        P = pool_groups.get(key)
        if P is None:
            unmatched += [{ID: i, "reason": "no private firm in same stratum"} for i in L[ID]]
            continue
        avail = P.assign(tiebreak=rng.random(len(P)))
        picks = {idx: [] for idx in L.index}
        for _ in range(MAX_CONTROLS):                       # round-robin
            for idx, row in L.iterrows():
                if avail.empty:
                    break
                gap = (avail["firm_age"] - row["firm_age"]).abs()
                ok = avail.assign(age_gap=gap)
                if max_age_gap is not None:
                    ok = ok[ok["age_gap"] <= max_age_gap]
                if ok.empty:
                    continue
                best = ok.sort_values(["age_gap", "tiebreak"]).index[0]
                picks[idx].append((best, gap[best]))
                avail = avail.drop(best)                    # no replacement
        for idx, row in L.iterrows():
            if not picks[idx]:
                reason = ("no unused private firm within age limit" if max_age_gap is not None
                          else "stratum's private firms all used")
                unmatched.append({ID: row[ID], "reason": reason})
                continue
            n_sets += 1
            sets.append({"set_id": n_sets, "role": "listed", ID: row[ID],
                         "firm_age": row["firm_age"], "age_gap": 0.0,
                         "log_assets": row["log_assets"], **{k: row[k] for k in KEYS}})
            for c_idx, c_gap in picks[idx]:
                c = P.loc[c_idx]
                sets.append({"set_id": n_sets, "role": "private", ID: c[ID],
                             "firm_age": c["firm_age"], "age_gap": c_gap,
                             "log_assets": c["log_assets"], **{k: c[k] for k in KEYS}})

    ml = pd.DataFrame(sets)
    ml["n_controls"] = ml["set_id"].map(ml[ml["role"] == "private"].groupby("set_id").size())
    return ml, pd.DataFrame(unmatched, columns=[ID, "reason"]), L_all, pool

# Main analysis: no age limit (as in the proposal)
matched_long, unmatched, listed_df, pool = run_matching(eda)
rng = np.random.default_rng(607)      # used by the tests below

n_ctrl = matched_long[matched_long["role"] == "private"].groupby("set_id").size()
print(f"Matched {matched_long['set_id'].nunique()} of {len(listed_df)} listed firms")
print("\nControls per set:\n", n_ctrl.value_counts().sort_index())
print("\nUnmatched reasons:\n", unmatched["reason"].value_counts())
print("\nAny private firm used twice?",
      matched_long.loc[matched_long["role"] == "private", ID].duplicated().any())

Matched 351 of 496 listed firms

Controls per set:
 1    111
2     54
3    186
Name: count, dtype: int64

Unmatched reasons:
 reason
no private firm in same stratum     90
stratum's private firms all used    55
Name: count, dtype: int64

Any private firm used twice? False


353 of 496 listed firms (71%) were matched. 89 had no private firm in the same industry, age band and account type, and 54 were in strata with more listed than private firms. Exact consolidation-code matching was infeasible (495 of 496 listed firms are C1, versus 1 private firm), so firms were matched on broad account type (C vs U).

## Matched gap and balance
D_i = ln(assets) of the listed firm − mean ln(assets) of its matched private firms.
Balance is checked on actual firm age before and after matching.

Dᵢ for each set

In [5]:
lst = matched_long[matched_long["role"] == "listed"].set_index("set_id")
prv = matched_long[matched_long["role"] == "private"]

sets_df = pd.DataFrame({
    "y_listed": lst["log_assets"],
    "y_private_mean": prv.groupby("set_id")["log_assets"].mean(),
    "n_controls": lst["n_controls"],
})
sets_df["D"] = sets_df["y_listed"] - sets_df["y_private_mean"]
sets_df["D"].describe().round(3)

count    351.000
mean       2.206
std        2.541
min       -7.931
25%        0.705
50%        2.074
75%        3.803
max       11.876
Name: D, dtype: float64

B: balance

In [6]:
def smd(a, b, wb=None):
    """Standardised mean difference; wb = weights for group b."""
    wb = np.ones(len(b)) if wb is None else np.asarray(wb)
    mb = np.average(b, weights=wb)
    vb = np.average((b - mb) ** 2, weights=wb)
    return (a.mean() - mb) / np.sqrt((a.var() + vb) / 2)

w_prv = 1 / prv["n_controls"]          # each matched set counts equally

balance = pd.DataFrame({
    "Listed mean age":  [listed_df["firm_age"].mean(), lst["firm_age"].mean()],
    "Private mean age": [pool["firm_age"].mean(),
                         np.average(prv["firm_age"], weights=w_prv)],
    "SMD (age)": [smd(listed_df["firm_age"], pool["firm_age"]),
                  smd(lst["firm_age"], prv["firm_age"], w_prv)],
}, index=["Before matching (all firms)", "After matching"]).round(3)
balance.to_csv(TAB_DIR / "matching_balance.csv")
print(balance)

print("\nAge gap to matched controls (years): median =", prv["age_gap"].median(),
      "| 90th pct =", prv["age_gap"].quantile(0.9), "| max =", prv["age_gap"].max())

# Matched vs unmatched listed firms
is_matched = listed_df[ID].isin(lst[ID])
print("\nMatched vs unmatched listed firms:")
print(listed_df.assign(matched=is_matched).groupby("matched")
      .agg(n=(ID, "size"), mean_age=("firm_age", "mean"),
           median_log_assets=("log_assets", "median")).round(2))

# Match rate by age band and top industries
rate = lambda col: (listed_df.assign(matched=is_matched)
                    .groupby(col, observed=True)["matched"]
                    .agg(n_listed="size", match_rate="mean").round(2))
print("\nMatch rate by age band:\n", rate("age_band"))
print("\nMatch rate, 10 largest industries:\n",
      rate("nace2").sort_values("n_listed", ascending=False).head(10))

                             Listed mean age  Private mean age  SMD (age)
Before matching (all firms)           24.591            11.283      0.874
After matching                        23.781            22.004      0.107

Age gap to matched controls (years): median = 1.0 | 90th pct = 12.0 | max = 90.0

Matched vs unmatched listed firms:
           n  mean_age  median_log_assets
matched                                  
False    145     26.55              18.38
True     351     23.78              18.39

Match rate by age band:
           n_listed  match_rate
age_band                      
0–4             26        0.96
5–9             62        0.85
10–19          164        0.62
20+            244        0.70

Match rate, 10 largest industries:
        n_listed  match_rate
nace2                      
68           38        1.00
26           38        0.47
46           31        1.00
94           27        0.04
10           20        1.00
28           19        0.95
47           18    

In [7]:
r = rate("nace2")
print("NACE 94:\n", r.loc[r.index == "94"])
print("\nLowest match rates among industries with ≥10 listed firms:\n",
      r[r["n_listed"] >= 10].sort_values("match_rate").head(5))

NACE 94:
        n_listed  match_rate
nace2                      
94           27        0.04

Lowest match rates among industries with ≥10 listed firms:
        n_listed  match_rate
nace2                      
94           27        0.04
61           10        0.10
30           14        0.21
24           13        0.23
42           13        0.38


Match rates vary by industry. Several small industries with only 1–3 listed firms (e.g. NACE 03, 05, 07, 90, 91) had no matches. Among industries with at least 10 listed firms, the lowest match rate was in NACE 94 (4%), [followed by electronics (NACE 26) at 47%].

## Matched inference
Following proposal Section 4(3): one-sided exact sign test, within-set permutation test of the median gap (9,999 permutations), and 95% percentile bootstrap CI for the median gap (5,000 resamples of whole matched sets).

one-sided sign test

In [8]:
D = sets_df["D"].to_numpy()
nz = D[D != 0]
k = int((nz > 0).sum())
sign = stats.binomtest(k, len(nz), p=0.5, alternative="greater")
print(f"H0: P(D > 0) = 0.5   vs   H1: P(D > 0) > 0.5")
print(f"Listed larger in {k}/{len(nz)} sets ({k/len(nz):.1%}), one-sided p = {sign.pvalue:.3g}")

H0: P(D > 0) = 0.5   vs   H1: P(D > 0) > 0.5
Listed larger in 300/351 sets (85.5%), one-sided p = 2.4e-44


The listed firm is larger than its matched private firms in 305 of 353 sets (86.4%; one-sided sign test p < 0.001). The median matched gap is 2.01 on the natural-log scale, i.e. the median listed firm is about 7.5 times larger than comparable private firms (within-set permutation test, one-sided p < 0.001; 95% bootstrap CI 5.9× to 10.5×). These results are associations, not causal effects of listing.

within-set permutation test of the median gap

In [9]:
obs_median = np.median(D)
N_PERM = 9_999
perm_D = np.empty((N_PERM, len(sets_df)))
col = 0
for m, grp in matched_long.groupby("n_controls"):
    Y = (grp.sort_values(["set_id", "role"])            # listed first within each set
            .groupby("set_id")["log_assets"].apply(np.array))
    Y = np.vstack(Y.to_numpy())                         # rows = sets, column 0 = listed
    total = Y.sum(axis=1)
    j = rng.integers(0, m + 1, size=(N_PERM, len(Y)))   # who gets the "listed" label
    yj = Y[np.arange(len(Y)), j]
    perm_D[:, col:col + len(Y)] = yj - (total - yj) / m
    col += len(Y)

perm_medians = np.median(perm_D, axis=1)
p_perm = (1 + np.sum(perm_medians >= obs_median)) / (N_PERM + 1)
print(f"Observed median gap = {obs_median:.3f} (≈ {np.exp(obs_median):.1f}× larger)")
print(f"One-sided permutation p = {p_perm:.4f}")

Observed median gap = 2.074 (≈ 8.0× larger)
One-sided permutation p = 0.0001


bootstrap CI for the median gap

In [10]:
B = 5_000
idx = rng.integers(0, len(D), size=(B, len(D)))      # resample whole matched sets
boot_medians = np.median(D[idx], axis=1)
lo, hi = np.percentile(boot_medians, [2.5, 97.5])
print(f"95% CI for median D: [{lo:.3f}, {hi:.3f}]")
print(f"→ the median listed firm is {np.exp(lo):.1f}× to {np.exp(hi):.1f}× larger than its matched private firms")

95% CI for median D: [1.803, 2.408]
→ the median listed firm is 6.1× to 11.1× larger than its matched private firms


## Matched quantile gaps
Δ_q = Q_listed(q) − Q_private(q) at q = 0.25, 0.50, 0.75, 0.90, using matched firms only, with equal total weight per matched set (each private control weighted 1/n_controls). 95% CIs from 5,000 matched-set bootstrap resamples.

In [11]:
def wquantile(x, w, q):
    """Weighted quantile (inverse-CDF definition)."""
    order = np.argsort(x)
    x, w = x[order], w[order]
    cw = np.cumsum(w) / w.sum()
    return x[np.searchsorted(cw, q)]

set_ids = sets_df.index.to_numpy()
yl = sets_df["y_listed"].to_numpy()
prv_by_set = {s: g["log_assets"].to_numpy() for s, g in prv.groupby("set_id")}

def gaps(pos, qs):
    L = yl[pos]
    P = np.concatenate([prv_by_set[set_ids[p]] for p in pos])
    W = np.concatenate([np.full(len(prv_by_set[set_ids[p]]),
                                1 / len(prv_by_set[set_ids[p]])) for p in pos])
    return np.array([np.quantile(L, q, method="inverted_cdf") - wquantile(P, W, q)
                     for q in qs])

QS = [0.25, 0.50, 0.75, 0.90]
point = gaps(np.arange(len(set_ids)), QS)
boot = np.array([gaps(rng.integers(0, len(set_ids), len(set_ids)), QS)
                 for _ in range(5_000)])
ci = np.percentile(boot, [2.5, 97.5], axis=0)

matched_qgaps = pd.DataFrame({
    "quantile": [f"Q{int(q*100)}" for q in QS],
    "ln_gap": point, "ci_low": ci[0], "ci_high": ci[1],
    "ratio": np.exp(point), "ratio_low": np.exp(ci[0]), "ratio_high": np.exp(ci[1]),
}).round(3)
matched_qgaps.to_csv(TAB_DIR / "matched_quantile_gaps.csv", index=False)
matched_qgaps

,quantile,ln_gap,ci_low,ci_high,ratio,ratio_low,ratio_high
0,Q25,2.125,1.857,2.416,8.375,6.406,11.203
1,Q50,2.055,1.718,2.340,7.807,5.576,10.383
2,Q75,2.358,1.918,2.641,10.569,6.808,14.028
3,Q90,2.141,1.672,2.755,8.508,5.324,15.718


unadjusted quantile gap

In [12]:
# Unadjusted quantile gaps (full sample, before matching), for comparison
L_all = eda.loc[eda["listing_status"] == "Listed", "log_assets"].to_numpy()
U_all = eda.loc[eda["listing_status"] == "Unlisted", "log_assets"].to_numpy()
raw = pd.DataFrame({
    "quantile": [f"Q{int(q*100)}" for q in QS],
    "ln_gap": [np.quantile(L_all, q) - np.quantile(U_all, q) for q in QS],
})
raw["ratio"] = np.exp(raw["ln_gap"])
raw.round(3).to_csv(TAB_DIR / "unadjusted_quantile_gaps.csv", index=False)
raw.round(3)

,quantile,ln_gap,ratio
0,Q25,5.362,213.124
1,Q50,4.620,101.472
2,Q75,4.441,84.862
3,Q90,4.328,75.818


After matching, the size gap is similar across the distribution (about 8–11× at Q25, Q50, Q75 and Q90), with heavily overlapping confidence intervals. This indicates a broad upward shift rather than an effect concentrated in the upper tail. Compared with the unadjusted gaps (roughly [A]–[B]×), the estimated size gap is substantially attenuated after matching, suggesting that differences in industry, age and consolidation status account for part of the raw gap. Because 145 listed firms could not be matched, the matched results apply to the common-support subset only, and they are not causal.

In [13]:
summary = pd.DataFrame({
    "statistic": ["Listed firms", "Matched sets", "Unmatched: no private firm in stratum",
                  "Unmatched: stratum exhausted", "Age SMD before", "Age SMD after",
                  "Sign test: share listed larger", "Sign test one-sided p",
                  "Median D (ln)", "Median ratio", "Permutation one-sided p",
                  "Bootstrap 95% CI low (ratio)", "Bootstrap 95% CI high (ratio)"],
    "value": [len(listed_df), len(sets_df),
              (unmatched["reason"] == "no private firm in same stratum").sum(),
              (unmatched["reason"] == "stratum's private firms all used").sum(),
              balance.iloc[0]["SMD (age)"], balance.iloc[1]["SMD (age)"],
              round(k / len(nz), 3), f"{sign.pvalue:.3g}",
              round(obs_median, 3), round(np.exp(obs_median), 2), f"{p_perm:.4f}",
              round(np.exp(lo), 2), round(np.exp(hi), 2)],
})
summary.to_csv(TAB_DIR / "matched_inference_summary.csv", index=False)
summary

,statistic,value
0,Listed firms,496
1,Matched sets,351
2,Unmatched: no private firm in stratum,90
3,Unmatched: stratum exhausted,55
4,Age SMD before,0.874
5,Age SMD after,0.107
6,Sign test: share listed larger,0.855
7,Sign test one-sided p,2.4e-44
8,Median D (ln),2.074
9,Median ratio,7.96


## Robustness: maximum age gap
The main analysis has no limit on the age gap within a stratum (max gap 90 years). As a robustness check, matching is repeated requiring controls to be within 15 and 10 years of the listed firm.

In [14]:
def median_gap_summary(ml, B=5_000, seed=607):
    r = np.random.default_rng(seed)
    lst = ml[ml["role"] == "listed"].set_index("set_id")["log_assets"]
    prv = ml[ml["role"] == "private"].groupby("set_id")["log_assets"].mean()
    D = (lst - prv).to_numpy()
    boot = np.median(D[r.integers(0, len(D), size=(B, len(D)))], axis=1)
    lo_, hi_ = np.percentile(boot, [2.5, 97.5])
    return {"matched_sets": len(D), "share_listed_larger": round((D > 0).mean(), 3),
            "median_ratio": round(np.exp(np.median(D)), 2),
            "ci_low": round(np.exp(lo_), 2), "ci_high": round(np.exp(hi_), 2)}

rows = {"Main (no age limit)": median_gap_summary(matched_long)}
for cap in [15, 10]:
    ml_cap, _, _, _ = run_matching(eda, max_age_gap=cap)
    rows[f"Age gap ≤ {cap} years"] = median_gap_summary(ml_cap)

robust_age = pd.DataFrame(rows).T
robust_age.to_csv(TAB_DIR / "robustness_age_gap.csv")
robust_age

,matched_sets,share_listed_larger,median_ratio,ci_low,ci_high
Main (no age limit),351.0,0.855,7.96,6.07,11.23
Age gap ≤ 15 years,341.0,0.848,7.13,5.48,10.65
Age gap ≤ 10 years,338.0,0.855,6.87,5.35,9.36


Restricting controls to within 15 or 10 years of age leaves the conclusion unchanged: the median listed firm remains about 7× larger than its matched private firms (7.13× with a 15-year limit, 6.87× with a 10-year limit, compared with 7.96× in the main analysis), with overlapping 95% confidence intervals, and the listed firm is larger in about 85% of sets in all three specifications.